In [1]:
import pandas as pd
import pickle
import os

# Loading data

In [ ]:
ROOT = '../../results/v3'

fn = os.path.join(ROOT, 'names', 'chinese', 'CNKI_uniqueChineseGivenNames_CNKIpop.csv')

fn_n = os.path.join(ROOT, 'gender', 'tool_n', 'Dict_gender_all.pkl')
fn_r = os.path.join(ROOT, 'gender', 'tool_r', 'CNKI_uniqueChineseGivenNames_withNGNU.csv')
fn_w = os.path.join(ROOT, 'gender', 'tool_w', 'CNKI_uniqueChineseGivenNames_g.csv')

fn_n_new = os.path.join(os.path.dirname(fn_n), 'CNKI_uniqueChineseGivenNames_gender.csv')
fn_r_new = os.path.join(os.path.dirname(fn_r), 'CNKI_uniqueChineseGivenNames_gender.csv')
fn_w_new = os.path.join(os.path.dirname(fn_w), 'CNKI_uniqueChineseGivenNames_gender.csv')

## Names

In [3]:
df_names = pd.read_csv(fn)
df_names.shape

(39712, 4)

In [4]:
df_names.head(2)

,givenname,pinyin,english,CNKI_pop
0,卫民,wèimín,weimin,17
1,丽英,lìyīng#####líyīng,liying,22


## N
Namsor

In [5]:
with open(fn_n, 'rb') as f:
    data_n = pickle.load(f)

data_n = {k: v for k, v in data_n.items() if isinstance(v, dict)}

df_n = pd.DataFrame.from_dict(data_n, orient="index")
df_n.index.name = "givenname" 
df_n.reset_index(inplace=True)
df_n.givenname.nunique()

40303

In [6]:
df_n.head(2)

,givenname,script,id,name,likelyGender,genderScale,score,probabilityCalibrated
0,家泗,HAN,c024ffed-f7bc-4627-8735-1652314ef056,潘%家泗,male,-0.252559,2.030027,0.626280
1,中城,HAN,4cf52dbb-0f61-4239-ae31-2a8062d2ec9d,沈%中城,male,-0.380731,4.350568,0.690365


## R

In [7]:
df_r = pd.read_csv(fn_r)
df_r.givenname.unique().shape

(39712,)

In [8]:
df_r.head(2)

,givenname,NLen,NU,NG
0,卫民,2,2.4869,0.6418
1,丽英,2,1.6550,-0.9308


## W

In [9]:
df_w = pd.read_csv(fn_w, index_col=0)
df_w.givenname.unique().shape

(39713,)

In [10]:
df_w.head(2)

,givenname,pinyin_wacc,pinyin,english,gender
0,俊池,Jùn Chí,jùnchí,junchi,male
1,建军,Jiàn Jūn,jiànjūn,jianjun,male


# Curating Data

In [11]:
# Chinese name in <tool>, but not in Names
for tool, df in [('R', df_r), ('N', df_n), ('W', df_w)]:
    print('')
    remove_from_tool = set(df.givenname.unique()) - set(df_names.givenname.unique())

    if len(remove_from_tool) == 0:
        print(f'No names to remove from {tool}.')
        continue

    print('before: ', tool, df.shape, len(remove_from_tool), list(remove_from_tool)[:min(5, len(remove_from_tool))])

    # removing chinese names in tool that are not in Names
    df.drop(df[df.givenname.isin(remove_from_tool)].index, inplace=True)
    print('final: ', tool, df.shape)


No names to remove from R.

before:  N (40303, 8) 592 ['', '友陆', '恒通', '尚賢', '荣洁']
final:  N (39711, 8)

before:  W (39713, 5) 2 ['格尔', nan]
final:  W (39711, 5)


In [12]:
df_r.shape, df_n.shape, df_w.shape

((39712, 4), (39711, 8), (39711, 5))

In [13]:
# Chinese name in Names, but not in tool
for tool, df in [('R', df_r), ('N', df_n), ('W', df_w)]:
    print('')
    add_to_tool = set(df_names.givenname.unique()) - set(df.givenname.unique())

    if len(add_to_tool) == 0:
        print(f'No names to add to {tool}.')
        continue

    print('before: ', tool, df.shape, len(add_to_tool), list(add_to_tool)[:min(5, len(add_to_tool))])

    if list(add_to_tool)[0] == '屿':
        
        # adding chinese names

        if tool == 'N':
            tmp = pd.DataFrame({'givenname': list(add_to_tool), 
                                'script': 'HAN',
                                'id': '8903ca9e-d3b7-45e3-9a28-0759ce469826',
                                'name': None,
                                'likelyGender': 'female',
                                'genderScale': 0.9672835688021866,
                                'score': 9.756042876470197,
                                'probabilityCalibrated': 0.9836417844010933})
        elif tool == 'W':
            tmp = pd.DataFrame({'givenname': list(add_to_tool), 
                                'pinyin_wacc': 'HAN',
                                'pinyin': 'yǔ',
                                'english': 'yu',
                                'gender': 'male'
                                })
        elif tool == 'R':
            print(' CHECK R ')

    if tool == 'R':
        print(' CHECK R ')
    elif tool == 'N':
        df_n = pd.concat([df_n, tmp], ignore_index=True)
        df = df_n
    elif tool == 'W':
        df_w = pd.concat([df_w, tmp], ignore_index=True)
        df = df_w
        
    print('after: ', tool, df.shape)


No names to add to R.

before:  N (39711, 8) 1 ['屿']
after:  N (39712, 8)

before:  W (39711, 5) 1 ['屿']
after:  W (39712, 5)


In [14]:
df_r.shape, df_n.shape, df_w.shape

((39712, 4), (39712, 8), (39712, 5))

In [15]:
# Double check (both empty)
set(df_n.givenname.unique()) - set(df_r.givenname.unique()), 
set(df_r.givenname.unique()) - set(df_n.givenname.unique()),
set(df_w.givenname.unique()) - set(df_r.givenname.unique()), 
set(df_r.givenname.unique()) - set(df_w.givenname.unique())

set()

In [16]:
df_n.sort_values('givenname').to_csv(fn_n_new, index=False)
df_r.sort_values('givenname').to_csv(fn_r_new, index=False)
df_w.sort_values('givenname').to_csv(fn_w_new, index=False)